# Week 1: Research Question and Provisional Lane

**Assignment:** ML-02 · **Track:** Machine Learning · **Phase:** Setup

This notebook frames a provisional capstone lane. It uses the anonymized starter dataset only, and the recommendation described here is for human review rather than automatic content changes.

## 1. My lane and why

I am choosing **Lane 2: Refresh / Content Opportunity Scoring**. The lane will produce a ranked, explainable queue of pages that a content team should review first for a possible refresh, expansion, protection, pruning, or monitoring decision.

I chose this lane because the starter data contains page-level search visibility, content freshness, position, click-through rate, and engagement signals. Those signals can help sort a large inventory into a manageable review order. The starter notebooks also showed that a learned ranking can be compared with a transparent rule, which makes this a practical place to begin while keeping the recommendation inspectable.

This is a **provisional** choice. For the capstone, I expect to strengthen the target from the starter dataset's current-window decline proxy to a future-window outcome, subject to the approved warehouse release and a leakage review.

## 2. The question: decision, action, and cost of a wrong call

**Search question.** Given a limited weekly review capacity, which pages should a content team review first because they show the strongest observed evidence of a visibility or content opportunity?

**Unit of analysis.** One row is one anonymized content item, or page, observed over the starter dataset's trailing 90-day window. `client_id` is used only for grouping and later validation, never as a model feature.

**Output.** A ranked decision-support queue. Each row should include a priority score, a suggested review action, a small set of reason codes, and a confidence note. The queue is not an instruction to edit a page automatically.

**Decision and action.** A content strategist or SEO reviewer would take the top pages that fit their capacity and inspect them. After human review, they could choose to refresh or expand content, protect a valuable page, investigate a CTR issue, monitor it, or decide that no change is justified.

**Cost of a wrong recommendation.** A false positive spends scarce review and editing time on a page that does not need that attention. A false negative leaves a worthwhile opportunity lower in the queue, which can delay a useful intervention. Neither error proves lost or gained traffic, so the scoring policy should match the team's capacity and tolerate uncertainty.

**Why data or ML can help.** The inventory is too large to inspect page by page without prioritization. A transparent rule provides a baseline. If a leakage-safe model consistently improves the ordering on an appropriate held-out evaluation, it can help reviewers spend their limited attention on more promising candidates. This is not simply "train a model": the useful product is a justified decision, an actionable queue, and an explanation of why each page was surfaced.

## 3. Quick look at the data

The following cells load the shipped starter dataset and calculate three supporting observations. I use `impressions_90d >= 500` only as a simple, transparent visibility threshold for this early framing. It is a policy choice to revisit, not a universal definition of importance.

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [root / "data/raw/content_refresh_anonymized.csv" for root in [Path.cwd(), *Path.cwd().parents]]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv. Run from the repository root or work/notebooks/.")

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} page-level rows and {df['client_id'].nunique()} pseudonymized clients.")

Loaded 30,000 page-level rows and 32 pseudonymized clients.


In [2]:
is_declining = df["trend_direction"].eq("down")
is_visible = df["impressions_90d"].ge(500)
visible_and_declining = is_visible & is_declining

print(f"1. Starter scale: {len(df):,} pages across {df['client_id'].nunique()} pseudonymized clients.")
print(f"2. Current decline proxy: {is_declining.sum():,} pages ({is_declining.mean():.1%}) are labelled 'down'.")
print(f"3. Reviewable visible-decline pool: {visible_and_declining.sum():,} pages are both 'down' and have at least 500 impressions in 90 days.")

1. Starter scale: 30,000 pages across 32 pseudonymized clients.
2. Current decline proxy: 16,262 pages (54.2%) are labelled 'down'.
3. Reviewable visible-decline pool: 9,961 pages are both 'down' and have at least 500 impressions in 90 days.


These observations support the lane choice. The data has a meaningful page inventory, the current decline proxy identifies a large potential review pool, and even a simple visibility threshold still leaves far more candidates than a team can inspect manually. A ranking and an explicit capacity policy are therefore more useful than a flat list.

The `trend_direction == "down"` field is used here only to describe the starter dataset's current proxy. It will not be used as a feature, and neither will `trend_pct`, because it encodes the same outcome.

## 4. Careful words: what I can and cannot claim

- I can say that this anonymized starter slice contains observed signals associated with a current decline proxy and that a ranked queue may help organize human review.
- I cannot say that a recommendation will cause a page to recover, that an edit will increase traffic, or that I have proved a Google ranking factor. Those claims need an experiment or a stronger causal design.
- The starter label is derived from a current comparison window. It is useful for learning the workflow, but it is not a future forecast. A later version should use a clearly separated feature window and future target window.
- Before modelling, I will audit timing and leakage, exclude `trend_direction` and `trend_pct` from features, and use validation that keeps related client pages or time periods appropriately separated.
- Low-volume movement, seasonality, consolidation, and changes in search-result presentation can all make a page look like an opportunity. The final queue must retain reason codes and require human review.

## 5. Self-check

- [x] I selected one predefined lane: Refresh / Content Opportunity Scoring.
- [x] I stated the research question, unit of analysis, output, decision, action, and wrong-call costs.
- [x] I loaded the starter CSV in code and showed three measured observations.
- [x] I explained why the work is a decision-support workflow, not just model training.
- [x] I recorded important limits, including current-window proxy labels, leakage, and non-causal claims.
- [x] I will revisit the lane and data contract before the end of Week 4.